<span style="color:#246485;font-weight:700;font-size:32px"> 
Option Analysis & Scoring Pipeline
</span> <br>

#### **architected by --> Alf Haugen**
-----

#### **Overview:** 
This notebook runs the end-to-end options analysis and scoring pipeline; 
from raw API data capture, premium and risk data build and scoring to a final 
scored universe structured for dashboard visualzation. 

The pipeline scans a universe of ~600 NYSE and Nasdaq symbols on a user-defined date, 
building a structured dataset of options premiums, implied volatility, historical volatility, 
price spike activity, and metrics across the forward term structure. 
All metrics are then scored and ranked into a two-dimensional premium/risk framework for put-selling alpha opportunities. 

#### **Objectives:** 
- Pull historical options chain and daily time-series price data from Alpha Vantage by symbol
- Build normalized put premium metrics across 5 DTE windows (Days to Expiration) 2-week, 1-month, 45-day, then the next two future
  LEAPS (14, 30, 45, over60_1, over60_2) and 4 delta-based strike buckets (ATM, Slight, Moderate, Far)
- Compute ATM straddle premium and three premium efficiency metrics
  (vs theoretical move, vs IV, vs realized vol)
- Compute Historical Volatility (HV_20, HV_30, HV_60) and IV/HV ratios per DTE window
- Compute spike analysis across 30 and 60-day windows with inward symbol and
  universal relative signals, producing a blended spike score based on frequency and
  magnitude
- Compute relative volatility vs SPY and QQQ benchmarks
- Score the universe on two axes — Premium Score and Risk Score,
  using weighted composites of the above metrics
- Assign each symbol to one of four quadrants (Q1 High Premium/Low Risk → Q4) for visualization
  and analysis
- Compute term structure slopes for premium and IV across DTE windows
- Output a single flat CSV per run for dashboard consumption

#### **Dataset Configurations:** 
- **Data source:**       Alpha Vantage Premium API (HISTORICAL_OPTIONS + TIME_SERIES_DAILY)
- **Option date:**       option_date  — the historical chain snapshot date (e.g. '2026-02-27')
- **HV as-of date:**     as_of_date   — price history for look-back analysis, truncated to prevent lookahead
- **Universe:**          ~600 symbols across NYSE and Nasdaq (ticker_list)
- **DTE windows:**       14-day (Friday-snapped), 30-day, over60_1, over60_2
- **Strike buckets:**    Delta based at: ATM (delta 0.40-0.60), Slight (0.25-0.40),
                         Moderate (0.15-0.25), Far (0.05-0.15)
- **Rate limit:**        75 calls/min by Alpha Vantage → batches of 37 symbols with 61s pause between batches
- **Benchmarks:**        SPY and QQQ HV_30 fetched once before the loop for relative vol

#### **Model Architecture Overview** 
The pipeline runs in four layers:

  1. API Layer (av_api_calls.py)
     Fetches options chain and daily prices per symbol. Manages rate limiting,
     error handling, and benchmark HV computation for SPY/QQQ.

  2. Premium Layer (option_prem_iv_builder.py)
     Parses and filters contracts by delta bucket and liquidity.
     Computes normalized premium (extrinsic/spot), aggregates by DTE/bucket,
     builds ATM straddle, and computes three premium efficiency metrics.

  3. Risk Layer (hist_vol_iv_risk_builder.py)
     Computes HV across three windows, extracts ATM IV per DTE window,
     builds IV/HV ratios with categorical signals, and runs spike analysis
     across 30 and 60-day windows with self-relative and universe signals.

  4. Scoring Layer (score_universe.py)
     Standardizes and weights premium and risk components into composite scores.
     Computes term structure regression slopes, premium efficiency signals,
     universe-relative spike and HV percentile ranks, and assigns quadrant labels.
     Outputs the final scored master DataFrame.

See ReadMe for further details on overall scoring model design. 

#### **Expected Results** 
- option_analysis_master  : flat DataFrame, one row per symbol, ~100+ columns
                            covering all premium, IV, HV, spike, and efficiency metrics.
                            ---> Pushes into Scoring Layer 
- scored_master           : option_analysis_master enriched with premium_score,
                            risk_score, quadrant, term structure slopes,
                            prem_efficiency_signal per DTE, blended spike_signal_universe,
                            HV_30_pct, relative_vol_spy_pct, relative_vol_qqq_pct
- error_log_df            : DataFrame of failed symbols with error message and
                            AV response detail for diagnosis
- CSV output              : scored_master saved to disk for dashboard consumption,
                            filename includes the option_date for run tracking

---

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from datetime import datetime, timedelta
import time
from IPython.display import clear_output
from typing import Optional
%config InlineBackend.figure_format = 'retina'
from scipy import stats

import requests
import json
import os
import dotenv
dotenv.load_dotenv()
import sys
import traceback
sys.tracebacklimit = 0 # turn off the error tracebacks

### reload amended source code / .py
%load_ext autoreload
%autoreload 2

In [2]:
os.getcwd()
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(project_root)

src_path = os.path.join(project_root, 'src')
sys.path.insert(0, src_path)

print(src_path)  # verify path looks right

/Users/AlfHaugen/Python/Code/options-alpha-scanner
/Users/AlfHaugen/Python/Code/options-alpha-scanner/src


In [3]:
### Imports
import av_api_calls
import score_universe

In [4]:
### symbols
path_one = src_path + '/tickers.txt'
# path_one = src_path + '/tickers_one_hun.txt'
# path_one = src_path + '/tickers_two_hun.txt'
# path_one = src_path + '/tickers_six_hun.txt'

with open(path_one, 'r') as f:
    ticker_list = [item.strip() for item in f.read().split(',')]

print(f' The number of symbols is {len(ticker_list)}')
ticker_list[-5:]

 The number of symbols is 10


['TQQQ', 'META', 'TSLA', 'CB', 'RJF']

In [7]:
# Setting up request and params. 
API_KEY = os.getenv('MktPremiumKey')
API_KEY = 

In [8]:
#### Data Params --->
option_date = '2026-04-10'   # date for historical options chain
as_of_date  = '2026-04-10'   # truncate HV history at this date

#### **Naming Convention**

In [9]:
### Naming --> 
file_size = '10'
data_as_of = as_of_date
opt_file_name = 'opt_analysis_master_' + file_size + '_' + data_as_of + '.csv'
error_log_file_name = 'error_log_' + file_size + '_' + data_as_of + '.csv'
stats_log_file_name = 'stats_desc_' + file_size + '_' + data_as_of + '.csv'
field_error_file_name = 'field_errors_' + file_size + '_' + data_as_of + '.csv'
opt_file_name

'opt_analysis_master_10_2026-04-10.csv'

--------
### **Run the API / Option Scanner Data Build**

----

In [10]:
### Call Option Scanner -->
option_analysis_master, error_log_df = av_api_calls.option_analysis_scan(ticker_list, API_KEY, option_date, as_of_date)

Scanning 10 symbols
[10/10] Current: RJF
Succeeded : 9
Failed    : 0
API calls : 18
Elapsed   : 00:20
 Pushing in the spot value 146.19 
Building Option Premium Data
  ⚠️ No expiration found near over60_1 DTE for RJF
  ⚠️ No expiration found near over60_1 DTE for RJF
Building Historical Vol and Spike Data
Building Straddle Data
OK
------------------------------------------

Done: 10 succeeded, 0 failed
Total time: 00:21
Master shape: (10, 113)


In [11]:
print(option_analysis_master.shape)
option_analysis_master.head(5)

(10, 113)


,symbol,date,spot,expiration_14,premium_atm_14,iv_atm_14,premium_slight_14,iv_slight_14,premium_moderate_14,iv_moderate_14,premium_far_14,iv_far_14,expiration_30,premium_atm_30,iv_atm_30,premium_slight_30,iv_slight_30,premium_moderate_30,iv_moderate_30,premium_far_30,iv_far_30,expiration_45,premium_atm_45,iv_atm_45,premium_slight_45,iv_slight_45,premium_moderate_45,iv_moderate_45,premium_far_45,iv_far_45,expiration_over60_1,premium_atm_over60_1,iv_atm_over60_1,premium_slight_over60_1,iv_slight_over60_1,premium_moderate_over60_1,iv_moderate_over60_1,premium_far_over60_1,iv_far_over60_1,expiration_over60_2,premium_atm_over60_2,iv_atm_over60_2,premium_slight_over60_2,iv_slight_over60_2,premium_moderate_over60_2,iv_moderate_over60_2,premium_far_over60_2,iv_far_over60_2,straddle_14,put_atm_14,call_atm_14,prem_per_iv_primary_14,prem_per_iv_sec_14,prem_per_hv30_14,straddle_30,put_atm_30,call_atm_30,prem_per_iv_primary_30,prem_per_iv_sec_30,prem_per_hv30_30,straddle_45,put_atm_45,call_atm_45,prem_per_iv_primary_45,prem_per_iv_sec_45,prem_per_hv30_45,straddle_over60_1,put_atm_over60_1,call_atm_over60_1,prem_per_iv_primary_over60_1,prem_per_iv_sec_over60_1,prem_per_hv30_over60_1,straddle_over60_2,put_atm_over60_2,call_atm_over60_2,prem_per_iv_primary_over60_2,prem_per_iv_sec_over60_2,prem_per_hv30_over60_2,HV_20,HV_30,HV_60,atm_iv_14,ratio_14,spread_14,signal_14,atm_iv_30,ratio_30,spread_30,signal_30,atm_iv_45,ratio_45,spread_45,signal_45,atm_iv_over60_1,ratio_over60_1,spread_over60_1,signal_over60_1,atm_iv_over60_2,ratio_over60_2,spread_over60_2,signal_over60_2,spike_count_30,spike_ratio_30,avg_spike_pct_30,max_spike_pct_30,spike_signal_30,spike_count_60,spike_ratio_60,avg_spike_pct_60,max_spike_pct_60,spike_signal_60,relative_vol_spy,relative_vol_qqq
0,GME,2026-04-10,23.22,2026-04-24,1.8411,31.2430,0.9475,28.8040,0.7752,34.6580,0.3804,43.1127,2026-05-15,3.1438,34.6580,1.9595,34.6580,1.2489,39.5360,0.4881,46.6897,2026-05-15,3.1438,34.6580,1.9595,34.6580,1.2489,39.5360,0.4881,46.6897,2026-06-18,5.7709,40.9990,3.7145,41.9745,2.1533,44.4140,0.9205,50.9988,2026-07-17,6.0724,40.1863,4.1021,37.5840,2.4009,43.4380,1.0192,47.9907,3.78,1.84,1.94,0.5132,0.0589,0.0670,5.60,3.14,2.45,0.4335,0.0907,0.1144,5.60,3.14,2.45,0.4335,0.0907,0.1144,11.19,5.77,5.42,0.5214,0.1408,0.2100,10.56,6.07,4.49,0.4213,0.1511,0.2209,0.3103,0.2749,0.3754,0.3124,1.137,0.0376,Equiv. Vol,0.3466,1.261,0.0717,Rich Vol,0.3466,1.261,0.0717,Rich Vol,0.4100,1.492,0.1351,Rich Vol,0.4019,1.462,0.1270,Rich Vol,1,0.73,3.75,3.75,Normal,3,1.10,6.56,7.93,Normal,1.54,1.29
1,AGQ,2026-04-10,122.21,2026-04-24,7.1598,111.8684,5.4558,116.0217,2.5315,122.4602,1.7320,129.2897,2026-05-15,10.7985,114.6558,9.3384,115.4687,5.1065,119.4118,2.0047,117.0945,2026-05-15,10.7985,114.6558,9.3384,115.4687,5.1065,119.4118,2.0047,117.0945,2026-06-18,14.3521,119.8357,13.8980,111.8138,7.2605,114.2808,3.0406,118.2033,2026-09-18,19.8397,121.3488,23.9075,113.6802,12.7786,111.0784,5.0732,112.5420,11.96,7.16,4.80,0.4536,0.0640,0.0589,15.78,10.80,4.98,0.3693,0.0942,0.0888,15.78,10.80,4.98,0.3693,0.0942,0.0888,18.73,14.35,4.37,0.2986,0.1198,0.1180,22.48,19.84,2.64,0.2318,0.1635,0.1632,1.1929,1.2159,2.4496,1.1187,0.920,-0.0972,Equiv. Vol,1.1466,0.943,-0.0694,Equiv. Vol,1.1466,0.943,-0.0694,Equiv. Vol,1.1984,0.986,-0.0176,Equiv. Vol,1.2135,0.998,-0.0024,Equiv. Vol,1,0.73,17.98,17.98,Normal,2,0.73,63.86,91.40,Normal,6.82,5.70
2,QQQ,2026-04-10,611.07,2026-04-24,1.1956,19.1181,0.8960,21.1498,0.4985,22.8533,0.2162,26.2026,2026-05-15,1.9009,20.0241,1.5351,23.1365,0.8956,26.1096,0.4378,30.9696,2026-05-15,1.9009,20.0241,1.5351,23.1365,0.8956,26.1096,0.4378,30.9696,2026-06-18,2.5693,20.3027,2.2251,23.1946,1.3169,26.2027,0.5934,31.1889,2026-07-17,3.3312,21.0486,2.6754,23.2463,1.6931,25.9703,0.7206,31.8810,2.32,1.20,1.12,0.5138,0.0625,0.0560,3.69,1.90,1.79,0.4946,0.0949,0.0891,3.69,1.90,1.79,0.4946,0.0949,0.0891,5.12,2.57,2.55,0.4820,0.1265,0.1204,7.07,3.33,3.74,0.5383,0.1583,0.1562,0.2343,0.2133,0.1947,0.1912,0.896,-0.0221,Co

-----
#### **Error Log and DF Stats**
- Review Results logged during the API Loop Run

In [ ]:
### Reviewing the Errors logged
print(error_log_df.shape)
error_log_df.head(10)

In [ ]:
### Save to CSV -->
error_log_df.to_csv(error_log_file_name, index=True)

#### **Statistics Review**

In [ ]:
### Run Description Stats
data_describe = option_analysis_master.describe().T
# data_describe.head()

In [ ]:
### Save to CSV -->
data_describe.to_csv(stats_log_file_name, index=True)

#### **Field Specific Review**

In [ ]:
field_nan_report = av_api_calls.audit_non_numeric(option_analysis_master)
print(field_nan_report['symbol'].unique())
field_nan_report.head(5)

In [ ]:
### Save to CSV --> 
field_nan_report.to_csv(field_error_file_name, index=True)

----
### **Run block to Save Premium / Risk Data to CSV**

In [ ]:
### Write to CSV -->
option_analysis_master.to_csv(opt_file_name, index=False)

<hr style="border:7px solid #246485">

## **Scoring Module**
- Push the Option Premium and Historical Vol. Builder dataframe into the Scoring module. 

In [ ]:
### If loading past data from saved CSV.....
# option_analysis_master = pd.read_csv('opt_analysis_master_200_sym_2026_02_27.csv')
# print(option_analysis_master.shape)
# option_analysis_master.head()

In [12]:
### Run data through Scoring Module: ----------->
scoring_data = score_universe.score_universe(option_analysis_master)

In [13]:
print(scoring_data.shape)
scoring_data.sort_values('risk_score',ascending=False).head(4)

(10, 138)


,symbol,date,premium_score,risk_score,quadrant,spot,expiration_14,premium_atm_14,iv_atm_14,premium_slight_14,iv_slight_14,premium_moderate_14,iv_moderate_14,premium_far_14,iv_far_14,expiration_30,premium_atm_30,iv_atm_30,premium_slight_30,iv_slight_30,premium_moderate_30,iv_moderate_30,premium_far_30,iv_far_30,expiration_45,premium_atm_45,iv_atm_45,premium_slight_45,iv_slight_45,premium_moderate_45,iv_moderate_45,premium_far_45,iv_far_45,expiration_over60_1,premium_atm_over60_1,iv_atm_over60_1,premium_slight_over60_1,iv_slight_over60_1,premium_moderate_over60_1,iv_moderate_over60_1,premium_far_over60_1,iv_far_over60_1,expiration_over60_2,premium_atm_over60_2,iv_atm_over60_2,premium_slight_over60_2,iv_slight_over60_2,premium_moderate_over60_2,iv_moderate_over60_2,premium_far_over60_2,iv_far_over60_2,straddle_14,put_atm_14,call_atm_14,prem_per_iv_primary_14,prem_per_iv_sec_14,prem_per_hv30_14,straddle_30,put_atm_30,call_atm_30,prem_per_iv_primary_30,prem_per_iv_sec_30,prem_per_hv30_30,straddle_45,put_atm_45,call_atm_45,prem_per_iv_primary_45,prem_per_iv_sec_45,prem_per_hv30_45,straddle_over60_1,put_atm_over60_1,call_atm_over60_1,prem_per_iv_primary_over60_1,prem_per_iv_sec_over60_1,prem_per_hv30_over60_1,straddle_over60_2,put_atm_over60_2,call_atm_over60_2,prem_per_iv_primary_over60_2,prem_per_iv_sec_over60_2,prem_per_hv30_over60_2,HV_20,HV_30,HV_60,atm_iv_14,ratio_14,spread_14,signal_14,atm_iv_30,ratio_30,spread_30,signal_30,atm_iv_45,ratio_45,spread_45,signal_45,atm_iv_over60_1,ratio_over60_1,spread_over60_1,signal_over60_1,atm_iv_over60_2,ratio_over60_2,spread_over60_2,signal_over60_2,spike_count_30,spike_ratio_30,avg_spike_pct_30,max_spike_pct_30,spike_signal_30,spike_count_60,spike_ratio_60,avg_spike_pct_60,max_spike_pct_60,spike_signal_60,relative_vol_spy,relative_vol_qqq,premium_slope,iv_slope,slope_divergence,wp_14,wp_30,wp_45,wp_over60_1,wp_over60_2,premium_slope_pct,iv_slope_pct,slope_div_pct,prem_efficiency_signal_14,prem_efficiency_signal_30,prem_efficiency_signal_45,prem_efficiency_signal_over60_1,prem_efficiency_signal_over60_2,spike_score_universe,spike_pct_universe,spike_signal_universe,HV_30_pct,relative_vol_spy_pct,relative_vol_qqq_pct
3,META,2026-04-10,-0.0446,1.3655,Q2 High Premium / High Risk,629.86,2026-04-24,2.0555,32.0970,1.3561,33.3570,0.7211,35.4707,0.3327,39.6749,2026-05-15,3.9936,40.3892,2.9553,42.0441,1.6095,44.0885,0.6954,48.3973,2026-05-15,3.9936,40.3892,2.9553,42.0441,1.6095,44.0885,0.6954,48.3973,2026-06-18,4.9194,36.9015,3.8629,38.4516,2.0759,40.6332,0.8920,45.1289,2026-07-17,5.6097,35.9583,4.4264,37.3892,2.4807,39.5358,1.0823,43.8716,4.01,2.06,1.96,0.5307,0.0640,0.0454,7.19,3.99,3.19,0.4776,0.0989,0.0882,7.19,3.99,3.19,0.4776,0.0989,0.0882,8.64,4.92,3.72,0.4476,0.1333,0.1087,9.96,5.61,4.35,0.4444,0.1560,0.1240,0.5342,0.4526,0.4385,0.3210,0.709,-0.1316,Compressed Vol,0.4039,0.892,-0.0487,Compressed Vol,0.4039,0.892,-0.0487,Compressed Vol,0.3690,0.815,-0.0835,Compressed Vol,0.3596,0.795,-0.0930,Compressed Vol,3,2.20,7.01,8.29,Moderate,4,1.47,7.74,9.90,Normal,2.54,2.12,0.040797,0.000137,0.040660,1.9156,3.7859,3.7859,4.7081,5.3730,0.6,0.45,0.6,Cheap + Thin,Cheap + Thin,Cheap + Thin,Cheap + Thin,Cheap + Thin,4.4860,1.0,Extreme,0.8,0.8,0.8
1,AGQ,2026-04-10,2.4971,0.4734,Q2 High Premium / High Risk,122.21,2026-04-24,7.1598,111.8684,5.4558,116.0217,2.5315,122.4602,1.7320,129.2897,2026-05-15,10.7985,114.6558,9.3384,115.4687,5.1065,119.4118,2.0047,117.0945,2026-05-15,10.7985,114.6558,9.3384,115.4687,5.1065,119.4118,2.0047,117.0945,2026-06-18,14.3521,119.8357,13.8980,111.8138,7.2605,114.2808,3.0406,118.2033,2026-09-18,19.8397,121.3488,23.9075,113.6802,12.7786,111.0784,5.0732,112.5420,11.96,7.16,4.80,0.4536,0.0640,0.0589,15.78,10.80,4.98,0.3693,0.0942,0.0888,15.78,10.80,4.98,0.3693,0.0942,0.0888,18.73,14.35,4.37,0.2986,0.1198,0.1180,22.48,19.84,2.64,0.2318,0.1635,0.1632,1.1929,1.2159,2.4496,1.1187,0.920,-0.0972,Equiv. Vol,1.1466,0.943,-0.0694,Equiv. Vol,1.1466,0.943,-0.0694,Equiv. Vol,1.1984,0.986,-0.0176,

#### **Save to CSV**

In [ ]:
### Write to CSV -->
opt_file_name = 'option_scores_master' + '_' + file_size + '_' + data_as_of + '.csv'
scoring_data.to_csv(opt_file_name, index=False)

### **End**
----